<div style="border-radius: 15px; border: 2px solid #6A1B9A; padding: 20px; background: linear-gradient(135deg, #9C27B0, #4CAF50); text-align: center; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);">
    <h1 style="color: #ffffff; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); font-weight: bold; margin-bottom: 10px; font-size: 36px; font-family: 'Roboto', sans-serif;">
        2️⃣⚡💡 LightGBM Duo: GBDT + GOSS+OOF 🔍
    </h1>
</div>

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">


### 📝 Updated Dataset Overview

The dataset for this project stems from the **Kaggle Playground Series - Season 4, Episode 12**, focusing on the prediction of insurance premiums. It encompasses a diverse range of features, simulating real-world scenarios in insurance premium determination.


#### **Key Highlights**:
- **Features**: A comprehensive set of **20 features** (excluding the target variable), including numerical, categorical, and temporal data types.
- **Target Variable**: The **Premium Amount**, a continuous numerical value, serves as the target for prediction.


#### **Feature Breakdown**:
1. **Numerical Features**:
   - Quantitative attributes such as:
     - **Age**: Reflecting policyholder age.
     - **Annual Income**: Indicating financial capability.
     - **Health Score**: A measure of the individual's health condition.
     - **Credit Score**: Evaluating financial reliability.
     - **Vehicle Age**: Representing the insured vehicle's age.
2. **Categorical Features**:
   - Qualitative characteristics, including:
     - **Gender**, **Marital Status**, **Education Level**, **Occupation**, and **Policy Type**, providing demographic and contextual insights.
3. **Temporal Feature**:
   - **Policy Start Date**: A date-based feature capturing the inception of the insurance policy.

#### **Dataset Challenges**:
1. **Missing Values**:
   - Several features contain missing data points, requiring thoughtful imputation techniques to maintain data integrity and enable robust predictions.
2. **Skewed Distributions**:
   - Features like **Annual Income** and **Premium Amount** exhibit significant skewness, necessitating transformations such as logarithmic scaling to normalize the data.
3. **Diverse Feature Types**:
   - The dataset includes a mix of numerical, categorical, and temporal data types, requiring comprehensive preprocessing strategies for consistency and model compatibility.
4. **Multicollinearity**:
   - Potential correlations among features may affect model performance, necessitating correlation analysis and feature selection.

### 🎯 **Objective**

The primary objective of this project is to build a **highly accurate and interpretable machine learning model** to predict the **Premium Amount** for insurance policyholders. The model will aim to balance predictive accuracy and usability for actionable insights.

#### **Key Objectives**:
1. **Data Preprocessing**:
   - Address missing values, normalize numerical distributions, and encode categorical variables for consistent and robust model training.
2. **Feature Engineering**:
   - Enhance predictive capability by:
     - Deriving new features from existing ones.
     - Selecting features with the highest predictive importance.
3. **Model Training and Optimization**:
   - Leverage cutting-edge models such as:
     - **LightGBM (GBDT and GOSS)** for gradient boosting.
   - Employ advanced tuning techniques like **Optuna** for hyperparameter optimization.
4. **Performance Evaluation**:
   - Measure model effectiveness using the **Root Mean Squared Logarithmic Error (RMSLE)** metric, ensuring alignment with real-world scenarios.
5. **Interpretability and Insights**:
   - Extract feature importance metrics to identify key drivers of premium costs.
   - Analyze prediction trends to uncover actionable insights for stakeholders.

#### **Project Aspirations**:
This project aims to:
- Achieve a **competitive RMSLE score**, surpassing baseline performance benchmarks.
- Provide a **scalable and adaptable framework** for insurance premium prediction, applicable across various datasets and contexts.
- Offer **meaningful insights** into the primary determinants of insurance premiums, aiding strategic decision-making.

By leveraging advanced **data science techniques**, **rigorous preprocessing**, and **state-of-the-art models**, this project seeks to establish itself as a reliable and innovative solution for predicting insurance premiums in the industry. 🚀

# <span style="color:transparent;">Import Libraries</span>

<div style="border-radius: 15px; border: 2px solid #6A1B9A; padding: 10px; background: linear-gradient(135deg, #9C27B0, #4CAF50); text-align: center; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);">
    <h1 style="color: #ffffff; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); font-weight: bold; margin-bottom: 5px; font-size: 28px; font-family: 'Roboto', sans-serif;">
        Import Libraries
    </h1>
</div>

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.cm import viridis
import seaborn as sns
import math
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from scipy.signal import find_peaks
from scipy.stats import skew 

import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from lightgbm import early_stopping, log_evaluation
from sklearn.model_selection import KFold

# Ignore general warnings
import warnings
warnings.filterwarnings("ignore")

# Suppress LightGBM logs
import logging
logging.getLogger("lightgbm").setLevel(logging.ERROR)


# <span style="color:transparent;">Data Loading and Initial Exploration</span>

<div style="border-radius: 15px; border: 2px solid #6A1B9A; padding: 10px; background: linear-gradient(135deg, #9C27B0, #4CAF50); text-align: center; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);">
    <h1 style="color: #ffffff; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); font-weight: bold; margin-bottom: 5px; font-size: 28px; font-family: 'Roboto', sans-serif;">
        Data Loading and Initial Exploration
    </h1>
</div>

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">


In [ ]:
# Load the datasets
train_data = pd.read_csv('/kaggle/input/playground-series-s4e12/train.csv',index_col=[0])
test_data = pd.read_csv('/kaggle/input/playground-series-s4e12/test.csv',index_col=[0])
sample_data = pd.read_csv('/kaggle/input/playground-series-s4e12/sample_submission.csv')

# Verify shapes
print("Train Data Shape:", train_data.shape)
print("Test Data Shape:", test_data.shape)

In [ ]:
# Display sample data
print("Training Dataset: \n")
display(train_data.head())
print('\n')
print("Test Dataset: \n")
display(test_data.head())

# <span style="color:transparent;">Data Inspection and Understanding</span>

<div style="border-radius: 15px; border: 2px solid #6A1B9A; padding: 10px; background: linear-gradient(135deg, #9C27B0, #4CAF50); text-align: center; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);">
    <h1 style="color: #ffffff; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); font-weight: bold; margin-bottom: 5px; font-size: 28px; font-family: 'Roboto', sans-serif;">
        Data Inspection and Understanding
    </h1>
</div>

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">


In [ ]:
# Display information for the training dataset
print("Training Dataset Information: \n")
train_info = train_data.info()
display(train_info)
print('\n')
# Display information for the test dataset
print("Test Dataset Information: \n")
test_info = test_data.info()
display(test_info)

#### Dataset Shapes:
1. **Train Dataset**: Contains 1,200,000 rows and 20 columns.
2. **Test Dataset**: Contains 800,000 rows and 19 columns.
   - The `Premium Amount` column, which is the target variable, is missing in the test dataset (as expected).

#### Data Types:
1. **Train Dataset**: 
   - 9 numerical columns (`float64`) and 11 categorical/text columns (`object`).
2. **Test Dataset**:
   - 8 numerical columns (`float64`) and 11 categorical/text columns (`object`).


# <span style="color:transparent;">Handling Missing Data</span>

<div style="border-radius: 15px; border: 2px solid #6A1B9A; padding: 10px; background: linear-gradient(135deg, #9C27B0, #4CAF50); text-align: center; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);">
    <h1 style="color: #ffffff; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); font-weight: bold; margin-bottom: 5px; font-size: 28px; font-family: 'Roboto', sans-serif;">
        Handling Missing Data
    </h1>
</div>

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">


## Missing Values Overview and Heatmaps

In [ ]:
# Check for missing values in the datasets
missing_train = train_data.isnull()
missing_test = test_data.isnull()

# Create a single figure with subplots for both datasets
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Heatmap for missing values in the training dataset
sns.heatmap(missing_train, cmap='viridis', cbar=True, yticklabels=False, ax=axes[0])
axes[0].set_title('Missing Values Heatmap - Training Dataset', fontsize=14)
axes[0].set_xlabel('Features', fontsize=12)
axes[0].set_ylabel('Entries', fontsize=12)

# Heatmap for missing values in the test dataset
sns.heatmap(missing_test, cmap='viridis', cbar=True, yticklabels=False, ax=axes[1])
axes[1].set_title('Missing Values Heatmap - Test Dataset', fontsize=14)
axes[1].set_xlabel('Features', fontsize=12)
axes[1].set_ylabel('Entries', fontsize=12)

plt.tight_layout()
plt.show()


#### Insights from Missing Values Heatmaps:

1. **Training Dataset:**
   - Several features in the training dataset have missing values, as indicated by the yellow streaks in the heatmap.
   - Features such as `Number of Dependents`, `Occupation`, `Previous Claims`, and `Credit Score` have a relatively high number of missing values.
   - Some columns, such as `Policy Start Date` and `Gender`, seem to have no missing values as indicated by their continuous dark bands.

2. **Test Dataset:**
   - The test dataset also has missing values, with patterns similar to the training dataset.
   - Features like `Previous Claims`, `Occupation`, and `Number of Dependents` show significant missing values, aligning with the training dataset's pattern.
   - No additional features have missing values compared to the training dataset, which ensures consistency between the datasets.

3. **Dataset Comparison:**
   - The distribution of missing values appears consistent across both the training and test datasets, implying similar data collection or preprocessing methods were used.
   - The percentage of missing values for features like `Credit Score` and `Previous Claims` might influence their impact on predictive models.

In [ ]:
# Function to calculate missing values, percentages, and data types
def missing_values_table(df):
    missing_count = df.isnull().sum()
    missing_percentage = 100 * missing_count / len(df)
    data_types = df.dtypes
    return pd.DataFrame({
        'Missing Values': missing_count,
        'Percentage (%)': missing_percentage,
        'Data Type': data_types
    })

# Create tables for train and test datasets
train_missing_table = missing_values_table(train_data)
test_missing_table = missing_values_table(test_data)

# Display the tables
print("Missing Values Table - Training Dataset:\n")
display(train_missing_table[train_missing_table['Missing Values'] > 0])  # Display only features with missing values
print("\n")

print("Missing Values Table - Test Dataset:\n")
display(test_missing_table[test_missing_table['Missing Values'] > 0])  

#### **Observational Insights from Missing Values Table:**

#### Training Dataset:
1. **Columns with High Missing Values:**
   - `Occupation` (29.84%) and `Previous Claims` (30.34%) have the highest percentage of missing values. These features might significantly impact the dataset's completeness and require careful handling.
   - `Credit Score` (11.49%) and `Number of Dependents` (9.14%) also have notable missing percentages.

2. **Columns with Moderate Missing Values:**
   - Features such as `Health Score` (6.17%) and `Customer Feedback` (6.49%) have moderate levels of missing values.

3. **Columns with Minimal Missing Values:**
   - Features like `Vehicle Age` (0.0005%) and `Insurance Duration` (0.00008%) have very few missing values, which can be easily imputed without much impact.

4. **Data Types:**
   - Features with missing values include both numerical (`float64`) and categorical (`object`) data types, indicating the need for distinct imputation strategies.


#### Test Dataset:
1. **Columns with High Missing Values:**
   - Similar to the training dataset, `Occupation` (29.89%) and `Previous Claims` (30.35%) exhibit the highest percentage of missing values.
   - `Credit Score` (11.43%) and `Number of Dependents` (9.14%) also show significant missingness.

2. **Columns with Moderate Missing Values:**
   - Features such as `Health Score` (6.18%) and `Customer Feedback` (6.53%) align closely with the training dataset in terms of missing values.

3. **Columns with Minimal Missing Values:**
   - Features like `Vehicle Age` (0.000375%) and `Insurance Duration` (0.00025%) have minimal missing data, mirroring the training dataset.

4. **Consistency:**
   - The percentages of missing values in the test dataset are highly consistent with the training dataset, making it easier to apply uniform imputation strategies.


In [ ]:
# Filter missing values for train and test datasets
train_missing = train_missing_table[train_missing_table['Missing Values'] > 0].sort_values(by='Percentage (%)', ascending=False)
test_missing = test_missing_table[test_missing_table['Missing Values'] > 0].sort_values(by='Percentage (%)', ascending=False)

# Set up the figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# Bar plot for train dataset
train_colors = cm.get_cmap('viridis', len(train_missing))(range(len(train_missing)))
axes[0].barh(train_missing.index, train_missing['Percentage (%)'], color=train_colors)
axes[0].set_title('Percentage of Missing Values (Train Data)', fontsize=12)
axes[0].set_xlabel('Percentage (%)', fontsize=10)
axes[0].set_ylabel('Features', fontsize=10)
axes[0].grid(axis='x', linestyle='--', alpha=0.6)
axes[0].invert_yaxis()  

# Bar plot for test dataset
test_colors = cm.get_cmap('viridis', len(test_missing))(range(len(test_missing)))
axes[1].barh(test_missing.index, test_missing['Percentage (%)'], color=test_colors)
axes[1].set_title('Percentage of Missing Values (Test Data)', fontsize=12)
axes[1].set_xlabel('Percentage (%)', fontsize=10)
axes[1].grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


#### Key Observations:
- **Consistency Across Datasets:** The percentage of missing values in features is highly consistent between the training and test datasets, simplifying the imputation strategy.
- **Critical Features:** High missing percentages in key features like `Previous Claims` and `Occupation` could impact model performance significantly if not addressed carefully.
- **Potential Strategies:**
  - Imputation: Use median or mean values for numerical features like `Credit Score` and `Number of Dependents`. For categorical features like `Occupation`, use the mode or `"Unknown"`.

In [ ]:
# Filter only features with missing values in the training dataset
features_with_missing = train_missing_table[train_missing_table['Missing Values'] > 0].index.tolist()

def analyze_nan_with_target_filtered(df, target_column, features):
    missing_analysis = {}
    
    for col in features:
        # Split the data into missing and non-missing subsets for the column
        missing_mask = df[col].isnull()
        non_missing_mask = ~missing_mask
        
        # Calculate statistics for Premium Amount
        stats = {
            "Missing Count": missing_mask.sum(),
            "Non-Missing Count": non_missing_mask.sum(),
            "Mean (Missing)": df.loc[missing_mask, target_column].mean(),
            "Mean (Non-Missing)": df.loc[non_missing_mask, target_column].mean(),
            "Median (Missing)": df.loc[missing_mask, target_column].median(),
            "Median (Non-Missing)": df.loc[non_missing_mask, target_column].median(),
            "Std Dev (Missing)": df.loc[missing_mask, target_column].std(),
            "Std Dev (Non-Missing)": df.loc[non_missing_mask, target_column].std(),
        }
        
        missing_analysis[col] = stats
    
    return pd.DataFrame(missing_analysis).T

# Perform the analysis for only features with missing values
missing_vs_premium_filtered = analyze_nan_with_target_filtered(train_data, "Premium Amount", features_with_missing)

# Display the results
print("Analysis of Missing Values with Target (Premium Amount):\n")
display(missing_vs_premium_filtered)


In [ ]:
# List of float-type columns with missing values
float_missing_columns = ['Age', 'Annual Income', 'Number of Dependents', 
                         'Health Score', 'Previous Claims', 'Vehicle Age', 
                         'Credit Score', 'Insurance Duration']

for column in float_missing_columns:
    # Drop NaN values for binning, but retain the NaN group separately
    valid_data = train_data[column].dropna()
    bins = 10  

    # Bin the non-NaN values
    binned_data = pd.cut(valid_data, bins)
    df = pd.DataFrame({
        column: binned_data,
        'Premium Amount': train_data.loc[valid_data.index, 'Premium Amount']
    })
    
    # Group by the binned column and calculate mean Premium Amount
    grouped = df.groupby(column, observed=True, dropna=False).agg(
        lambda x: np.expm1(np.log1p(x).mean())
    ).reset_index()
    
    # Add the NaN group separately
    nan_group_mean = np.expm1(np.log1p(train_data.loc[train_data[column].isnull(), 'Premium Amount']).mean())
    nan_group = pd.DataFrame({column: ['NaN'], 'Premium Amount': [nan_group_mean]})
    
    # Concatenate the NaN group with the grouped data
    grouped = pd.concat([grouped, nan_group], ignore_index=True)
    
    def label(x):
        if isinstance(x, float) or x == 'NaN':
            return x
        x = x.mid  # Get the midpoint of the interval
        s = int(np.floor(np.log10(x)))
        return int(round(x, -s+1))

    # Select the viridis color
    viridis_colors = viridis(range(256))
    line_color = viridis_colors[5]  

    # Plot the non-NaN bins as a line plot
    plt.plot(grouped[:-1]['Premium Amount'], marker='o', color=line_color, label='non-nan')

    # Plot the NaN bin as a bar
    plt.bar(len(grouped) - 1, grouped.iloc[-1]['Premium Amount'], color='red', label='nan')

    # Set x-ticks and labels
    plt.xticks(range(len(grouped)), labels=grouped[column].apply(label), fontsize=6)
    plt.title(f'Average Premium Amount vs {column}')
    plt.xlabel(column)
    plt.ylabel('Premium Amount')
    plt.legend(loc='upper right')
    plt.show()


In [ ]:
# List of object-type columns with missing values
object_missing_columns = ['Marital Status', 'Occupation', 'Customer Feedback']

for column in object_missing_columns:
    # Group by the column and calculate the average log-transformed Premium Amount
    grouped = train_data.groupby(column)['Premium Amount'].agg(
        lambda x: np.expm1(np.log1p(x).mean())  # Use log-transform to calculate mean
    ).reset_index()
    
    # Calculate the average log-transformed Premium Amount for missing (NaN) values
    nan_group_mean = np.expm1(np.log1p(train_data.loc[train_data[column].isnull(), 'Premium Amount']).mean())
    
    # Add a row for the NaN group
    nan_group = pd.DataFrame({column: ['NaN'], 'Premium Amount': [nan_group_mean]})
    grouped = pd.concat([grouped, nan_group], ignore_index=True)
    
    # Use viridis color palette for the bars
    viridis_colors = viridis(range(256))  
    bar_colors = [viridis_colors[5]] * (len(grouped) - 1) + ['red']  

    # Plot the bar chart
    plt.figure(figsize=(10, 6))
    plt.bar(grouped[column].astype(str), grouped['Premium Amount'], color=bar_colors)
    
    # Set labels and title
    plt.title(f'{column.upper()} Log-Transformed Average Premium Amounts')
    plt.ylabel('Average Premium Amount')
    plt.xlabel(column)
    plt.xticks(rotation=45, ha='right')  
    plt.show()


## Imputation Strategies for Numeric and Categorical Data

### Filling Missing Values in Numeric Columns

In [ ]:
numeric_columns = train_data.select_dtypes(include=['number']).columns

for col in numeric_columns:
    if col in test_data.columns:
        # Impute missing values with -1
        train_data[col].fillna(-1, inplace=True)
        test_data[col].fillna(-1, inplace=True)


### Filling Missing Values in Object Columns

In [ ]:
object_columns = train_data.select_dtypes(include=['object']).columns
for col in object_columns:
    if col in test_data.columns:
        train_data[col].fillna("Unknown", inplace=True)
        test_data[col].fillna("Unknown", inplace=True)


- For each categorical column:
    - Missing values in both the `train_data` and `test_data` for that column are replaced with the string `"Unknown"`.

In [ ]:
# Verify Missing Values

print("Missing Values After Imputation - Training Dataset:")
print(train_data.isnull().sum())

print("\nMissing Values After Imputation - Test Dataset:")
print(test_data.isnull().sum())

#### **Missing Values Successfully Imputed**
   - After executing the imputation logic for both numeric and categorical columns:
     - All missing values in the training dataset (`train_data`) have been successfully replaced. 
     - All missing values in the test dataset (`test_data`) have been successfully replaced.
   - There are **no remaining NaN values** in any column for either dataset.

#### **Key Observations**
   - **Numeric Columns:** Missing values were filled using the **median** value of the respective column in the training dataset. This approach ensures that the central tendency of the data is preserved while mitigating the impact of outliers.
   - **Categorical Columns:** Missing values were filled with the placeholder `"Unknown"`. This ensures that the absence of data is explicitly marked without distorting the distribution of existing categories.

In [ ]:
# Check for duplicate rows in the training dataset
train_duplicates = train_data.duplicated().sum()
print(f"\nNumber of duplicate rows in the training dataset: {train_duplicates}")

# Check for duplicate rows in the test dataset
test_duplicates = test_data.duplicated().sum()
print(f"Number of duplicate rows in the test dataset: {test_duplicates}")

After checking for duplicate rows in both datasets:
 - **Training Dataset (`train_data`)**: There are **0 duplicate rows** detected.
 - **Test Dataset (`test_data`)**: There are **0 duplicate rows** detected.

# <span style="color:transparent;">Exploratory Data Analysis (EDA)</span>

<div style="border-radius: 15px; border: 2px solid #6A1B9A; padding: 10px; background: linear-gradient(135deg, #9C27B0, #4CAF50); text-align: center; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);">
    <h1 style="color: #ffffff; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); font-weight: bold; margin-bottom: 5px; font-size: 28px; font-family: 'Roboto', sans-serif;">
        Exploratory Data Analysis (EDA)
    </h1>
</div>

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">


## Target Column Extraction and Visualizing Distribution

In [ ]:
target_column = (set(train_data.columns) - set(test_data.columns)).pop()

print(f"Target column: {target_column}")
print(f"Data type: {train_data[target_column].dtype}")


In [ ]:
# Custom colormap using viridis
viridis_cmap = cm.get_cmap("viridis")

def visualize_premium_amount_with_peaks(data, feature='Premium Amount'):
    plt.figure(figsize=(9, 4))

    # Histogram with KDE
    plt.subplot(1, 2, 1)
    ax = sns.histplot(data[feature], bins=30, kde=True, color=viridis_cmap(0.5))
    plt.title(f'Histogram of {feature} with KDE', fontsize=11)
    plt.xlabel(feature, fontsize=10)
    plt.ylabel('Frequency', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6) 

    # Extract KDE values to find peaks
    kde = sns.kdeplot(data[feature], ax=ax, color=viridis_cmap(0.7)).lines[0].get_data()
    kde_x, kde_y = kde[0], kde[1]
    peaks, _ = find_peaks(kde_y)

    # Highlight peaks
    for peak_idx in peaks:
        plt.plot(kde_x[peak_idx], kde_y[peak_idx], "ro")  # Red dots on peaks

    # Box Plot
    plt.subplot(1, 2, 2)
    sns.boxplot(x=data[feature], color=viridis_cmap(0.5))
    plt.title(f'Box Plot of {feature}', fontsize=11)
    plt.xlabel(feature, fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)  
    plt.tight_layout()
    plt.show()

visualize_premium_amount_with_peaks(train_data, feature='Premium Amount')


1. **Distribution Characteristics**
   - The `Premium Amount` shows a **right-skewed distribution**, meaning most of the values are concentrated towards lower premiums, with fewer instances of higher premium amounts.
   - The red peaks highlight the **modes** (local maxima), indicating clusters where specific premium amounts are more common. The highest mode appears around 1000, suggesting this is a prevalent premium value in the dataset.

2. **Outliers and Variability**
   - The box plot demonstrates that a significant number of premium values fall within a lower range, as shown by the interquartile range (IQR).
   - There are **many outliers** present, representing higher premium amounts that lie far beyond the upper whisker of the box plot. These outliers may require further analysis to understand their nature or consider their treatment in modeling (e.g., log transformation).

## Distribution Analysis of Numerical Features

In [ ]:
# Define columns to analyze
columns_to_analyze = train_data.select_dtypes(include=['number']).columns.drop('Premium Amount')

viridis_cmap = cm.get_cmap("viridis")
# Extract three colors from the colormap
viridis_colors = [viridis_cmap(0.3), viridis_cmap(0.5), viridis_cmap(0.8)]

fig, axes = plt.subplots(len(columns_to_analyze), 3, figsize=(25, len(columns_to_analyze) * 5))

for i, column in enumerate(columns_to_analyze):
    # Histogram for train_data
    sns.histplot(train_data[column], bins=30, kde=True, color=viridis_colors[0], ax=axes[i, 0])
    axes[i, 0].set_title(f'Distribution of {column} (Train)', fontsize=14)
    axes[i, 0].set_xlabel(column, fontsize=10)
    axes[i, 0].set_ylabel('Frequency', fontsize=10)
    axes[i, 0].grid(visible=True, linestyle='--', alpha=0.6)

    # Boxplot for train_data
    sns.boxplot(x=train_data[column], color=viridis_colors[1], ax=axes[i, 1])
    axes[i, 1].set_title(f'Boxplot of {column} (Train)', fontsize=14)
    axes[i, 1].set_xlabel(column, fontsize=10)
    axes[i, 1].grid(visible=True, linestyle='--', alpha=0.6)

    # Boxplot for test_data
    sns.boxplot(x=test_data[column], color=viridis_colors[2], ax=axes[i, 2])
    axes[i, 2].set_title(f'Boxplot of {column} (Test)', fontsize=12)
    axes[i, 2].set_xlabel(column, fontsize=10)
    axes[i, 2].grid(visible=True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
# Select numeric columns 
numeric_data = train_data.select_dtypes(include=['number'])

# Compute the correlation matrix
correlation_matrix = numeric_data.corr()
plt.figure(figsize=(8, 6))

# Create the heatmap
sns.heatmap(
    correlation_matrix, 
    annot=True, 
    fmt=".2f", 
    cmap='viridis',  
    cbar=True, 
    square=True,
    mask=np.triu(np.ones_like(correlation_matrix, dtype=bool)),  
    linewidths=0.5  
)

plt.title('Correlation Heatmap of Numerical Features (Excluding Target)', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()


**Low Correlation Between Features:**
   - The majority of the features exhibit very low correlation values (close to 0), indicating that they are weakly related or independent.
   - This suggests minimal multicollinearity, which is beneficial for model training as it reduces the risk of redundancy in predictive features.

**Key Observations:**
   - **Credit Score vs Annual Income:** Shows a slightly negative correlation (-0.20), suggesting that individuals with lower income might have slightly higher credit scores or vice versa.
   - **Premium Amount vs Previous Claims:** The correlation value is positive (0.05), indicating that individuals with higher previous claims tend to have slightly higher premium amounts.
   - **Health Score vs Premium Amount:** A weak positive correlation (0.01) suggests minimal influence of health score on premium amount.

**Premium Amount Correlation:**
   - Most features show minimal correlation with the target variable (`Premium Amount`), highlighting the potential for non-linear relationships that might require advanced modeling techniques (e.g., tree-based algorithms).

## Categorical Feature Analysis

In [ ]:
# Function to display barplot and pie chart for categorical columns
def plot_categorical_distribution(data, column_name):
    plt.figure(figsize=(12, 4))
    
    # Bar plot for categorical distribution
    plt.subplot(1, 2, 1)
    sns.countplot(y=column_name, data=data, palette='Set2')
    plt.title(f'Distribution of {column_name}', fontsize=12)
    plt.xlabel('Count', fontsize=10)
    plt.ylabel(column_name, fontsize=10)

    ax = plt.gca()
    for p in ax.patches:
        count = int(p.get_width())
        ax.annotate(f'{count}', 
                    (p.get_width() + 0.1, p.get_y() + p.get_height() / 2), 
                    ha='left', va='center', fontsize=10, color='black')
    
    sns.despine(left=True, bottom=True)
    
    # Pie chart for percentage distribution
    plt.subplot(1, 2, 2)
    data[column_name].value_counts().plot.pie(
        autopct='%1.1f%%', 
        colors=sns.color_palette('Set2', data[column_name].nunique()), 
        startangle=90, 
        explode=[0.05] * data[column_name].nunique(), 
        shadow=True
    )
    plt.title(f'Percentage Distribution of {column_name}', fontsize=12)
    plt.ylabel('')  

    plt.tight_layout()
    plt.show()

categorical_columns = ['Gender', 'Marital Status', 'Education Level', 'Occupation', 'Location', 
                'Policy Type', 'Customer Feedback', 'Smoking Status', 'Exercise Frequency', 'Property Type']

for column in categorical_columns:
    plot_categorical_distribution(train_data, column)

In [ ]:
# Function to calculate count and percentage of unique values
def unique_values_table(data, categorical_columns):
    results = {}
    for column in categorical_columns:
        value_counts = data[column].value_counts()
        percentages = (value_counts / len(data)) * 100
        results[column] = pd.DataFrame({
            'Value': value_counts.index,
            'Count': value_counts.values,
            'Percentage (%)': percentages.values
        })
    return results

# Specify categorical columns
categorical_columns = ['Gender', 'Marital Status', 'Education Level', 'Occupation', 'Location', 
                       'Policy Type', 'Customer Feedback', 'Smoking Status', 
                       'Exercise Frequency', 'Property Type']

# Get unique value tables for each categorical column
unique_values_results = unique_values_table(train_data, categorical_columns)

for column in categorical_columns[:10]:  
    print(f"Unique Values for {column}:\n")
    display(unique_values_results[column])


#### Observational Insights:

1. **Gender Distribution:**
   - The dataset is balanced with a nearly equal proportion of males (50.21%) and females (49.79%).

2. **Marital Status:**
   - The three primary categories (Single, Married, Divorced) are distributed almost equally, each accounting for about 33%.
   - A small percentage (1.54%) of data is categorized as `Unknown`, which could be imputed or treated based on the analysis.

3. **Education Level:**
   - Education levels are uniformly distributed among Bachelor's, Master's, PhD, and High School, with percentages ranging from 24% to 25%.
   - No major outlier category exists in education, making this feature consistent.

4. **Occupation:**
   - A significant portion (29.84%) of data falls under the `Unknown` category, indicating missing values that were replaced during preprocessing.
   - The remaining categories (Employed, Self-Employed, Unemployed) are evenly distributed around 23%.

5. **Location:**
   - The dataset shows an almost equal split between `Suburban`, `Rural`, and `Urban` areas, each contributing about 33% of the data.

6. **Policy Type:**
   - The types of policies (`Premium`, `Comprehensive`, `Basic`) are evenly distributed, with each type accounting for approximately one-third of the data.

7. **Customer Feedback:**
   - Feedback categories (`Average`, `Poor`, `Good`) account for around 31% each, with an `Unknown` category representing 6.48%.
   - The presence of `Unknown` suggests gaps in feedback collection that might need attention.

8. **Smoking Status:**
   - A balanced distribution exists between `Yes` (50.16%) and `No` (49.84%) categories, making it a useful feature for potential analysis.

9. **Exercise Frequency:**
   - The dataset is evenly split across all four categories (`Weekly`, `Monthly`, `Rarely`, `Daily`), each contributing about 25%.

10. **Property Type:**
    - The three property types (`House`, `Apartment`, `Condo`) are evenly distributed, each accounting for approximately 33%.

#### Key Observations:
- Many categorical features have balanced distributions, which can be advantageous for model training as it reduces bias.
- Certain features, such as `Occupation`, `Customer Feedback`, and `Marital Status`, contain `Unknown` categories due to imputed missing values. Their treatment depends on the modeling approach.
- Most features exhibit uniform distribution across their categories, suggesting no dominance of a single class, which could enhance feature diversity in the model.

## Categorical Features vs Premium Amount

In [ ]:
# List of categorical columns
categorical_columns = [
    'Gender', 'Marital Status', 'Education Level', 'Occupation', 'Location', 
    'Policy Type', 'Customer Feedback', 'Smoking Status', 'Exercise Frequency', 'Property Type'
]

# Loop through each categorical feature to display summary statistics and box plot
for column in categorical_columns:
    # Calculate summary statistics grouped by the categorical column
    stats = train_data.groupby(column)['Premium Amount'].agg(['mean', 'median', 'count'])
    
    # Display summary statistics
    print(f"\nSummary Statistics for Premium Amount by {column}:")
    print(stats)
    
    # Plot box plot
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=train_data, x=column, y='Premium Amount', palette='viridis')
    plt.title(f'Premium Amount by {column}', fontsize=12)
    plt.xlabel(column, fontsize=11)
    plt.ylabel('Premium Amount', fontsize=11)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# <span style="color:transparent;">Data Preprocessing</span>

<div style="border-radius: 15px; border: 2px solid #6A1B9A; padding: 10px; background: linear-gradient(135deg, #9C27B0, #4CAF50); text-align: center; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);">
    <h1 style="color: #ffffff; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); font-weight: bold; margin-bottom: 5px; font-size: 28px; font-family: 'Roboto', sans-serif;">
        Data Preprocessing
    </h1>
</div>

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">


## Converting Date Columns to Epoch Time

In [ ]:
# Retrieve columns with 'object' data type
datetime_columns = train_data.select_dtypes(include=['object']).columns

for col in datetime_columns:
    try:
        # Convert the column to datetime format
        train_data[col] = pd.to_datetime(train_data[col], errors='raise')
        test_data[col] = pd.to_datetime(test_data[col], errors='raise')
        
        # Convert datetime to epoch time (in seconds)
        train_data[col] = train_data[col].astype(np.int64) / 10**9
        test_data[col] = test_data[col].astype(np.int64) / 10**9

        print(f"Converted '{col}' to epoch time.")
    except Exception as e:
        print(f"Skipping column '{col}' due to: {e}")


In [ ]:
# Check the first few rows and data type of the 'Policy Start Date' column
column_name = 'Policy Start Date'

# Display the first few rows
print(f"Sample data for '{column_name}':\n", train_data[column_name].head())

# Display the data type of the column
print(f"Data type of '{column_name}' in train_data: {train_data[column_name].dtype}")

# Repeat for test_data
print(f"Sample data for '{column_name}' in test_data:\n", test_data[column_name].head())
print(f"Data type of '{column_name}' in test_data: {test_data[column_name].dtype}")


- Successfully converts the 'Policy Start Date' column to datetime and then to epoch time.
- The converted Policy Start Date values (e.g., 1.703345e+09) are epoch times in seconds.
- The float64 data type indicates that the values are now numerical.

## Label Encoding Categorical Features

In [ ]:
def identify_non_numerical_features(data, dataset_name):
    non_numerical_features = data.select_dtypes(include=['object'])
    print(f"Non-Numerical Features and Unique Values in {dataset_name} dataset:")
    for column in non_numerical_features.columns:
        unique_values = non_numerical_features[column].unique()
        print(f"\n{column}: {unique_values}")

# Apply the function to training and test datasets
identify_non_numerical_features(train_data, "Training")
print("\n")
identify_non_numerical_features(test_data, "Test")

In [ ]:
# Define the encoding strategies for specific features
binary_features = ['Gender', 'Smoking Status']
ordinal_features = {
    'Exercise Frequency': ['Rarely', 'Monthly', 'Weekly', 'Daily']
}
nominal_features = ['Marital Status', 'Education Level', 'Occupation', 
                    'Location', 'Policy Type', 'Customer Feedback', 'Property Type']

# Binary Encoding for binary features
le = LabelEncoder()
for feature in binary_features:
    train_data[feature] = le.fit_transform(train_data[feature])
    test_data[feature] = le.transform(test_data[feature])

# Ordinal Encoding for ordered features
for feature, order in ordinal_features.items():
    oe = OrdinalEncoder(categories=[order])
    train_data[feature] = oe.fit_transform(train_data[[feature]]).flatten()  # Flatten to 1D
    test_data[feature] = oe.transform(test_data[[feature]]).flatten()       # Flatten to 1D

# One-Hot Encoding for nominal features
train_data = pd.get_dummies(train_data, columns=nominal_features, drop_first=True)
test_data = pd.get_dummies(test_data, columns=nominal_features, drop_first=True)


## Verifying Data Types Across Datasets

In [ ]:
# Create data type tables for train_data and test_data
train_data_types = pd.DataFrame({
    'Column Name': train_data.columns,
    'Train Data Type': train_data.dtypes
})

test_data_types = pd.DataFrame({
    'Column Name': test_data.columns,
    'Test Data Type': test_data.dtypes
})

# Merge the two tables for comparison
data_types_comparison = pd.merge(
    train_data_types, 
    test_data_types, 
    on='Column Name', 
    how='outer'
)

# Display the data types comparison table
print("Data Types Comparison of Train and Test Datasets:\n")
display(data_types_comparison)


## Normalization of Numerical Features

In [ ]:
# Select numerical columns
numerical_columns = train_data.select_dtypes(include=['float64']).columns
numerical_columns = numerical_columns[numerical_columns != target_column]

# Applying Normalization
scaler = StandardScaler()
train_data[numerical_columns] = scaler.fit_transform(train_data[numerical_columns])
test_data[numerical_columns] = scaler.transform(test_data[numerical_columns])


- The StandardScaler is used to standardize numerical features by removing the mean and scaling to unit variance.
- The scaler is fit on the train_data to calculate the mean and standard deviation and then applied to both train_data and test_data.

#### Why This Step is Important:
- Normalization ensures that all numerical features contribute equally to the model, preventing features with larger magnitudes (e.g., `Annual Income`) from dominating those with smaller values (e.g., `Vehicle Age`).

## Distribution Analysis of Preprocessed Features

In [ ]:
# Identify numeric columns only (excluding boolean columns)
numeric_columns = train_data.select_dtypes(include=[np.number]).columns

# Calculate the number of rows and columns needed
num_features = len(numeric_columns)
num_cols = 4
num_rows = math.ceil(num_features / num_cols)

# Create subplots
fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(16, num_rows * 3))
viridis_cmap = cm.get_cmap('viridis', len(numeric_columns))

# Plot each numeric column
for i, column in enumerate(numeric_columns):
    ax = axes.flatten()[i]
    train_data[column].hist(
        ax=ax, 
        bins=20, 
        color=viridis_cmap(i / len(numeric_columns)),  
        edgecolor='black', 
        linewidth=0.5
    )
    ax.set_title(column, fontsize=9)
    ax.tick_params(axis='both', which='major', labelsize=6)
    ax.grid(True, linestyle='--', alpha=0.6)  

# Remove empty subplots if any
for j in range(i + 1, len(axes.flatten())):
    fig.delaxes(axes.flatten()[j])

plt.suptitle('Dataset Feature Distributions (train_data)', fontsize=11)
plt.tight_layout()
plt.show()

## Skewness Reduction with Log Transformation

In [ ]:
# Define the continuous columns
continuous_columns_train = ['Annual Income', 'Premium Amount']  
continuous_columns_test = ['Annual Income']  

# Calculate skewness for the specified continuous columns
train_skewness = train_data[continuous_columns_train].apply(skew)
test_skewness = test_data[continuous_columns_test].apply(skew)

# Display results
print("Skewness for Training Dataset:\n")
display(train_skewness)

print("\nSkewness for Test Dataset:\n")
display(test_skewness)

In [ ]:
# Log-transform skewed features
train_data['Annual Income'] = np.log1p(train_data['Annual Income'])
test_data['Annual Income'] = np.log1p(test_data['Annual Income'])

In [ ]:
# Select only numeric columns from the training data
numeric_data = train_data.select_dtypes(include=['number'])

# Add a new column for the log-transformed values of target variable
numeric_data['Log_Transformed_Premium'] = np.log1p(train_data[target_column])

#### **Explanation and Insights:**

#### Skewness Analysis:
1. **Training Dataset Skewness:**
   - `Annual Income`: A skewness value of **1.522952** indicates moderate positive skewness, meaning that the distribution is tailing to the right, with a significant number of smaller income values and fewer larger ones.
   - `Premium Amount`: A skewness value of **1.240914** also shows positive skewness, indicating that most premium values are clustered at the lower range with fewer higher premiums.

2. **Test Dataset Skewness:**
   - `Annual Income`: The skewness value of **1.516915** mirrors the training data, confirming that both datasets share similar distributions for this feature.

#### Log Transformation:
- Applying a log transformation to `Annual Income` reduces its skewness and compresses the range of larger values. This step makes the distribution closer to normal, benefiting algorithms sensitive to skewed data.
  
#### Adding Log of Target (`Log_Transformed_Premium`):
- The addition of a `Log_Transformed_Premium` column in the training dataset creates a normalized target variable that minimizes the impact of extreme outliers, leading to better model performance.

## Correlation Analysis of Preprocessed Data

In [ ]:
# Compute the correlation matrix
train_corr_matrix = numeric_data.corr()

# Create the heatmap
plt.figure(figsize=(12, 9))
sns.heatmap(
    train_corr_matrix, 
    annot=True, 
    cmap='viridis',   
    vmax=1, 
    vmin=-1,
    annot_kws={"size": 8}, 
    fmt=".3f",
    linewidths=0.5  
)

# Customize x and y tick labels
plt.xticks(rotation=80, fontsize=9)
plt.yticks(fontsize=9)

# Add title and layout adjustments
plt.title("Correlation Heatmap of the Train Data", fontsize=11)
plt.tight_layout()
plt.show()


#### Difference Between `Premium Amount` and `Log_Transformed_Premium` with Other Features:

1. **Correlation Strength:**
   - The correlation between `Log_Transformed_Premium` and other features tends to be slightly weaker compared to the correlation of the original `Premium Amount` with the same features. This is because the log transformation compresses the range of the `Premium Amount`, reducing the influence of extreme values (outliers).

2. **Normalization of Relationships:**
   - The log-transformed `Log_Transformed_Premium` shows more stable and normalized relationships with features like `Health Score` and `Customer Feedback`, which might indicate that these relationships are nonlinear in the raw `Premium Amount`.

3. **Reduction in Skewness Impact:**
   - In features like `Annual Income`, where the original correlation with `Premium Amount` is mildly negative, the correlation with `Log_Transformed_Premium` is further reduced. This suggests that the transformation reduces the skewed influence of high `Premium Amount` values.

4. **Stronger Predictive Relationships:**
   - The correlation between `Log_Transformed_Premium` and the original target (`Premium Amount`) is very high, indicating that the transformation retains most of the predictive information. However, the log-transformed target smooths out extreme values and better aligns with features like `Health Score` and `Previous Claims`.

5. **Impact of High-Variance Features:**
   - Features with high variability, such as `Credit Score` or `Annual Income`, exhibit more stable correlations with `Log_Transformed_Premium`, making them potentially more predictable in models using the transformed target.

6. **Feature Contribution Differences:**
   - The relationship between categorical features (e.g., `Location`, `Policy Type`) and the target remains nearly the same for `Premium Amount` and `Log_Transformed_Premium`, indicating that these features are less influenced by the target transformation.

7. **Higher Correlation with `Log_Transformed_Premium` for Log-Normal Relationships:**
   - Features like `Health Score` and `Policy Start Date` show slightly stronger positive correlations with `Log_Transformed_Premium` compared to `Premium Amount`, suggesting that these features have log-normal relationships with the target.

#### Key Observations:
- The log-transformation of the target variable (`Log_Transformed_Premium`) smooths the relationships between the target and input features, particularly for features influenced by outliers (e.g., `Annual Income` and `Previous Claims`).
- It reduces the magnitude of extreme correlations, creating a more stable predictive target while retaining the most critical relationships for predictive modeling.
- Features showing slight differences in correlation strength post-transformation indicate the utility of `Log_Transformed_Premium` for handling nonlinear relationships effectively.

## Separating Features and Target

In [ ]:
# Separate Features and Target
X_train = train_data.drop([target_column], axis=1)  # Features
y_train = train_data[target_column]                   # Target variable

## Applying Log Transformation to the target variable

In [ ]:
# Apply log transformation to the target variable
y_train_log = np.log1p(y_train)  # log1p is used for log(1 + x)

In [ ]:
# Custom colormap using viridis
viridis_cmap = cm.get_cmap("viridis")

# Select two colors from the colormap
color1 = viridis_cmap(0.5)  
color2 = viridis_cmap(0.3)  

# Plot with the custom colors
plt.figure(figsize=(9, 4))

# Plot original target distribution
plt.subplot(1, 2, 1)
sns.histplot(y_train, kde=True, bins=30, color=color1)
plt.title(f'Histogram of Target: {target_column} (y)', fontsize=11)
plt.xlabel(f'{target_column} (y)', fontsize=10)
plt.ylabel('Frequency', fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=7)
plt.grid(True, linestyle='--', alpha=0.6)

# Log-transformed target distribution
plt.subplot(1, 2, 2)
sns.histplot(y_train_log, kde=True, bins=30, color=color2)
plt.title('Histogram of log(y + 1)', fontsize=11)
plt.xlabel('log(y + 1)', fontsize=10)
plt.ylabel('Frequency', fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=7)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

#### Histogram of `log(y + 1)`
1. **Transformed Distribution:**
   - After applying the logarithmic transformation (`log(y + 1)`), the distribution becomes **approximately normal**, reducing the skewness observed in the original data.
   - This normalization ensures that most values fall within a manageable range for machine learning models, improving their performance.

2. **Symmetry and Centralization:**
   - The log transformation centers the distribution, with a peak around the range of `log(y + 1)` values between 6 and 7.


By compressing the scale of high premium values, the transformation mitigates the influence of outliers, reducing their weight in model training. A normal-like distribution aligns better with assumptions made by many regression models, potentially leading to more accurate predictions.

### Removing Whitespaces in Feature Names

In [ ]:
# Remove whitespaces in feature names for X_train and test_data
X_train.columns = X_train.columns.str.replace(' ', '_', regex=True)
test_data.columns = test_data.columns.str.replace(' ', '_', regex=True)

# Verify the updated column names
print("Updated Feature Names in X_train:")
print(X_train.columns)

print("\nUpdated Feature Names in test_data:")
print(test_data.columns)


### Validating Target and Transformed Target Data Types

In [ ]:
# Display data types
print("Feature Data Types (X_train):")
print(X_train.dtypes)

print("\nTarget Data Type (y_train):")
print(y_train.dtypes)

print("\nLog-Transformed Target Data Type (y_train_log):")
print(y_train_log.dtypes)


# <span style="color:transparent;">Model Training</span>

<div style="border-radius: 15px; border: 2px solid #6A1B9A; padding: 10px; background: linear-gradient(135deg, #9C27B0, #4CAF50); text-align: center; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);">
    <h1 style="color: #ffffff; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); font-weight: bold; margin-bottom: 5px; font-size: 28px; font-family: 'Roboto', sans-serif;">
        Model Training
    </h1>
</div>

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">


## Model Initialization

In [ ]:
# Model Initialization
lgb_gbdt = lgb.LGBMRegressor(
    boosting_type='gbdt',
    random_state=42,
    learning_rate=0.030362233382902903,
    n_estimators=998,
    max_depth=9,
    num_leaves=208,
    min_child_samples=11,
    subsample=0.8184667361186249,
    colsample_bytree=0.8616459477375787,
    reg_alpha=0.33080029457188864,
    reg_lambda=0.20736962602335904,
    objective='regression',
    metric='rmse',
    device='gpu',
    verbose=-1
)

lgb_goss = lgb.LGBMRegressor(
    boosting_type='goss',
    random_state=42,
    learning_rate=0.030362233382902903,
    n_estimators=998,
    max_depth=9,
    num_leaves=208,
    min_child_samples=11,
    subsample=0.8184667361186249,
    colsample_bytree=0.8616459477375787,
    reg_alpha=0.33080029457188864,
    reg_lambda=0.20736962602335904,
    objective='regression',
    metric='rmse',
    device='gpu',
    verbose=-1
)


- Both **lgb_gbdt** and **lgb_goss** models are correctly initialized with appropriate hyperparameters, including GPU usage via device='gpu'.

## OOF Predictions

In [ ]:
# Define the number of folds for OOF
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Initialize arrays to store OOF predictions and test predictions
oof_predictions_gbdt = np.zeros(len(X_train))
oof_predictions_goss = np.zeros(len(X_train))

test_predictions_gbdt = np.zeros((len(test_data), n_splits))
test_predictions_goss = np.zeros((len(test_data), n_splits))

# Store RMSLE for each fold
fold_rmsle_gbdt = []
fold_rmsle_goss = []

# OOF Training for LightGBM (GBDT) and LightGBM (GOSS)
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"Training Fold {fold + 1}/{n_splits}...")
    
    # Split the data into training and validation sets
    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_train_fold, y_val_fold = y_train_log.iloc[train_idx], y_train_log.iloc[val_idx]
    
    # LightGBM (GBDT)
    lgb_gbdt.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),  # Increased to 50
            lgb.log_evaluation(period=10)
        ]
    )
    oof_predictions_gbdt[val_idx] = lgb_gbdt.predict(X_val_fold)
    test_predictions_gbdt[:, fold] = lgb_gbdt.predict(test_data)
    fold_rmsle_gbdt.append(mean_squared_log_error(y_val_fold, oof_predictions_gbdt[val_idx]) ** 0.5)
    
    # LightGBM (GOSS)
    lgb_goss.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),  # Increased to 50
            lgb.log_evaluation(period=10)
        ]
    )
    oof_predictions_goss[val_idx] = lgb_goss.predict(X_val_fold)
    test_predictions_goss[:, fold] = lgb_goss.predict(test_data)
    fold_rmsle_goss.append(mean_squared_log_error(y_val_fold, oof_predictions_goss[val_idx]) ** 0.5)

# Compute average RMSLE for each model
avg_rmsle_gbdt = np.mean(fold_rmsle_gbdt)
avg_rmsle_goss = np.mean(fold_rmsle_goss)

print("Average RMSLE (GBDT):", avg_rmsle_gbdt)
print("Average RMSLE (GOSS):", avg_rmsle_goss)

In [ ]:
# Compute weights based on RMSLE
# Lower RMSLE => Higher Weight
total_weight = 1 / avg_rmsle_gbdt + 1 / avg_rmsle_goss
weight_gbdt = (1 / avg_rmsle_gbdt) / total_weight
weight_goss = (1 / avg_rmsle_goss) / total_weight

print("Weight for GBDT:", weight_gbdt)
print("Weight for GOSS:", weight_goss)

In [ ]:
# Compute weighted average of predictions
final_test_predictions = (
    weight_gbdt * test_predictions_gbdt.mean(axis=1) +
    weight_goss * test_predictions_goss.mean(axis=1)
)

# Exponentiate the final predictions (log scale to original scale)
final_test_predictions = np.expm1(final_test_predictions)

## Feature Importance

- Feature importance is calculated for both GBDT and GOSS models.
- Combined importance is averaged and plotted for the top 10 features, providing insights into the key drivers for predictions.

In [ ]:
# Combine feature importance for both GBDT and GOSS models
feature_importance_gbdt = pd.DataFrame({
    'Feature': X_train.columns,
    'GBDT Importance': lgb_gbdt.feature_importances_,
})

feature_importance_goss = pd.DataFrame({
    'Feature': X_train.columns,
    'GOSS Importance': lgb_goss.feature_importances_,
})

# Merge the feature importance for comparison
combined_feature_importance = pd.merge(
    feature_importance_gbdt,
    feature_importance_goss,
    on='Feature',
    how='inner'
)

# Sort features by average importance
combined_feature_importance['Avg Importance'] = (
    combined_feature_importance['GBDT Importance'] +
    combined_feature_importance['GOSS Importance']
) / 2

combined_feature_importance = combined_feature_importance.sort_values(
    by='Avg Importance', ascending=False
)

# Plot the feature importance
plt.figure(figsize=(12, 8))
sns.barplot(
    x='Avg Importance', 
    y='Feature', 
    data=combined_feature_importance.head(10),  # Top 10 features
    palette='viridis'
)
plt.title('Top 10 Features - Average Importance (GBDT & GOSS)', fontsize=16)
plt.xlabel('Average Importance Score', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Display top 10 features
print("Top 10 Features by Average Importance:\n")
display(combined_feature_importance.head(10))


## Prediction Distribution

- Histograms compare the distributions of the true values (y_train_log) and OOF predictions (oof_predictions_gbdt, oof_predictions_goss).
- This visualizes how well the models capture the target variable's distribution.

In [ ]:
# Calculate average predictions for test data
avg_test_predictions = (
    weight_gbdt * test_predictions_gbdt.mean(axis=1) +
    weight_goss * test_predictions_goss.mean(axis=1)
)

# Exponentiate predictions (log scale to original scale)
avg_test_predictions_exp = np.expm1(avg_test_predictions)

# Visualization of Prediction Distributions
viridis_cmap = cm.get_cmap("viridis", 3)

plt.figure(figsize=(12, 6))

# Plot true values (Train Data)
plt.hist(
    y_train_log, bins=30, color=viridis_cmap(0), alpha=0.6, edgecolor="black", label="True Values (Train)"
)

# Plot predicted values (OOF Predictions)
plt.hist(
    oof_predictions_gbdt, bins=30, color=viridis_cmap(0.5), alpha=0.6, edgecolor="black", label="Predicted Values (GBDT)"
)

plt.hist(
    oof_predictions_goss, bins=30, color=viridis_cmap(0.7), alpha=0.6, edgecolor="black", label="Predicted Values (GOSS)"
)

# Add titles and labels
plt.title("Prediction Distributions - Train and OOF Predictions", fontsize=16)
plt.xlabel("Log-transformed Premium Amount (log(y + 1))", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()


# <span style="color:transparent;">Creating the Submission File</span>

<div style="border-radius: 15px; border: 2px solid #6A1B9A; padding: 10px; background: linear-gradient(135deg, #9C27B0, #4CAF50); text-align: center; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);">
    <h1 style="color: #ffffff; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); font-weight: bold; margin-bottom: 5px; font-size: 28px; font-family: 'Roboto', sans-serif;">
        Creating the Submission File
    </h1>
</div>

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">


In [ ]:
# Prepare the submission file
submission = pd.DataFrame({"id": test_data.index, target_column: final_test_predictions})
submission.to_csv("ensemble_oof_submission.csv", index=False)
print("Submission file created successfully!")

In [ ]:
print(submission.head(10))

<div style="border-radius: 15px; border: 2px solid #6A1B9A; padding: 20px; background: linear-gradient(135deg, #9C27B0, #4CAF50); text-align: center; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);">
    <h1 style="color: #ffffff; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); font-weight: bold; margin-bottom: 10px; font-size: 28px; font-family: 'Roboto', sans-serif;">
        🙏 Thanks for Reading! 🚀
    </h1>
    <p style="color: #ffffff; font-size: 18px; text-align: center;">
        If you found this helpful, please upvote and share your thoughts!
    </p>
    <p style="color: #ffffff; font-size: 18px; text-align: center;">
        Happy Coding! 🙌😊
    </p>
</div>

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">
